### TODO:
1. Load multiple dataset for different buildings.
2. Get the climate info for the state.
3. preprocess the data.
4. Create a linear regression model.
5. Prepare some tests for the model.

In [184]:
import polars.selectors as cs
import polars as pl
import pyarrow
import pandas as pd
import numpy as np

In [185]:
from pathlib import Path
folder= Path().resolve().parent /'input'
meta_data_folder=Path().resolve().parent / 'metadata'

### 1. Load multiple dataset for different buildings.

In [186]:
# read the original data for three individual buildings
data=pl.scan_parquet(folder / '*')

In [187]:
# extract the buildings Ids
bldgs=data.select(pl.col('bldg_id').unique()).collect().to_series().to_list()

### 2. Get the climate info for the state.

In [188]:
pl.scan_parquet(meta_data_folder/'MetaData.parquet').collect().row(1, named=True)

{'bldg_id': 10,
 'completed_status': 'Success',
 'upgrade': 28,
 'in.upgrade_name': 'HVAC Demand Flexibility - Pre-peak Load Shifting, 2F Offset, 1hr Pre-peak Duration',
 'weight': 253.90367272727272,
 'applicability': True,
 'in.sqft..ft2': 1228.0,
 'in.representative_income': 32414.0,
 'in.county_name': 'Miami-Dade County',
 'in.ahs_region': 'CBSA Miami-Fort Lauderdale-West Palm Beach, FL',
 'in.aiannh_area': 'No',
 'in.area_median_income': '30-60%',
 'in.ashrae_iecc_climate_zone_2004': '1A',
 'in.ashrae_iecc_climate_zone_2004_sub_cz_split': '1A - FL',
 'in.bathroom_spot_vent_hour': 'Hour2',
 'in.battery': 'None',
 'in.bedrooms': '3',
 'in.building_america_climate_zone': 'Hot-Humid',
 'in.cec_climate_zone': 'None',
 'in.ceiling_fan': 'Standard Efficiency',
 'in.census_division': 'South Atlantic',
 'in.census_division_recs': 'South Atlantic',
 'in.census_region': 'South',
 'in.city': 'FL, Miami',
 'in.clothes_dryer': 'Electric',
 'in.clothes_dryer_usage_level': '100% Usage',
 'in.clot

In [189]:
%%time
# extract the climate zone details based on the builings and cast the building id type to match the one in the original data
Meta_data=pl.scan_parquet(meta_data_folder/'MetaData.parquet').with_columns(
    pl.col('bldg_id').cast(pl.UInt32)).filter(
    pl.col('bldg_id').is_in(bldgs
                           )).select(pl.col('in.occupants').cast(pl.Int32),'in.state','in.county',
                        'in.representative_income','in.area_median_income','in.income',
                        'in.income_recs_2020','in.income_recs_2015',
                        'in.building_america_climate_zone','in.ashrae_iecc_climate_zone_2004_sub_cz_split',
                        pl.col('in.bedrooms').cast(pl.Int32),'in.tenure','in.household_has_tribal_persons','bldg_id').unique()

CPU times: user 372 μs, sys: 0 ns, total: 372 μs
Wall time: 383 μs


In [190]:
%%time
# merge the climate zone changes into the original data
df=data.join(Meta_data, on='bldg_id')

CPU times: user 45 μs, sys: 0 ns, total: 45 μs
Wall time: 51 μs


In [191]:
df=df.rename({'in.ashrae_iecc_climate_zone_2004_sub_cz_split': 'climatezone',
                'out.electricity.cooling.energy_consumption..kwh': 'out.electricity.AC.energy_consumption..kwh'}).with_columns(
            pl.col('timestamp').dt.weekday().alias('day of the week'),
            pl.col('timestamp').dt.hour().alias('hour of the day'),
            pl.col('timestamp').dt.day().alias('day of the month'),
            pl.col('timestamp').dt.ordinal_day().alias('day of the year'),
            pl.col('timestamp').dt.week().alias('week of the year'),
            pl.col('timestamp').dt.month().alias('month of the year'),
            pl.col('timestamp').dt.quarter().alias('quarter')).with_columns(
            pl.when(pl.col('day of the week').is_in([6,7])).then(pl.lit("Yes")).otherwise(pl.lit("No")).alias('IsWeekend')
            ).select(
    cs.matches('^out.electricity.*|^out.site_energy.*|^bldg*|^day*|^hour*|^week*|^month*|^time*|^quarter|^IsWeekend|^in.*|^Short|^climatezone$')
).drop_nulls().collect()

In [192]:
# df=df.rename({'in.ashrae_iecc_climate_zone_2004_sub_cz_split': 'climatezone'}).with_columns(
#             pl.col('timestamp').dt.weekday().alias('day of the week'),
#             pl.col('timestamp').dt.hour().alias('hour of the day'),
#             pl.col('timestamp').dt.day().alias('day of the month'),
#             pl.col('timestamp').dt.ordinal_day().alias('day of the year'),
#             pl.col('timestamp').dt.week().alias('week of the year'),
#             pl.col('timestamp').dt.month().alias('month of the year'),
#             pl.col('timestamp').dt.quarter().alias('quarter')).with_columns(
#             pl.when(pl.col('day of the week').is_in([6,7])).then(pl.lit("Yes")).otherwise(pl.lit("No")).alias('is_weekend')
#             ).select(
#     cs.matches('^out.electricity.*|^out.site_energy.*|^bldg*|^day*|^hour*|^week*|^month*|^time|^quarter|^is_weekend|^in*|^climatezone$|^in.bedrooms')
# ).drop_nulls().collect()


----------------------------------------------

### Label Encoder for categorical variables

In [193]:
from sklearn.preprocessing import LabelEncoder
def encoding_categories(x):
    le=LabelEncoder()
    for dtype,col in zip(x.dtypes,x.columns):
        if dtype==pl.String:
            x=x.with_columns(pl.Series(col, le.fit_transform(x[col])))
    return x

## Preprocess the data

In [194]:
# defining x and y
def prepare_observations(d):
    x=d.select(cs.matches('^day*|^hour*|^week*|^month*|^time*|^quarter|^IsWeekend|^in.*|^Short|^climatezone$').exclude('in.sqft'),
            pl.col('out.electricity.total.energy_consumption..kwh').alias('total_usage')).with_columns(pl.col('timestamp').dt.timestamp()).select(pl.all().exclude('timestamp'))
    # x=d.select('climatezone','day of the week','hour of the day','day of the month','day of the year',
    #            'week of the year','month of the year','quarter','is_weekend','in.bedrooms','timestamp').with_columns(pl.col('timestamp').dt.timestamp())
    y=d.select(cs.matches('^out.electricity.*..kwh$').exclude('out.electricity.total.energy_consumption..kwh'))
    # y=d.select('out.electricity.heating.energy_consumption..kwh','out.electricity.cooling.energy_consumption..kwh')
    x=encoding_categories(x)
    return x,y

x,y=prepare_observations(df)
x

in.occupants,in.state,in.county,in.representative_income,in.area_median_income,in.income,in.income_recs_2020,in.income_recs_2015,in.building_america_climate_zone,climatezone,in.bedrooms,in.tenure,in.household_has_tribal_persons,day of the week,hour of the day,day of the month,day of the year,week of the year,month of the year,quarter,IsWeekend,total_usage
i32,i64,i64,f32,i64,i64,i64,i64,i64,i64,i32,i64,i64,i8,i8,i8,i16,i8,i8,i8,i64,f32
4,0,6,32414.0,4,3,1,1,0,0,3,0,0,1,0,1,1,1,1,1,0,0.1662
4,0,6,32414.0,4,3,1,1,0,0,3,0,0,1,0,1,1,1,1,1,0,0.15641
4,0,6,32414.0,4,3,1,1,0,0,3,0,0,1,0,1,1,1,1,1,0,0.14339
4,0,6,32414.0,4,3,1,1,0,0,3,0,0,1,1,1,1,1,1,1,0,0.14249
4,0,6,32414.0,4,3,1,1,0,0,3,0,0,1,1,1,1,1,1,1,0,0.14133
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
4,0,5,129716.0,3,0,0,0,0,1,4,0,0,1,23,31,365,1,12,4,0,0.47071
4,0,5,129716.0,3,0,0,0,0,1,4,0,0,1,23,31,365,1,12,4,0,0.22021
4,0,5,129716.0,3,0,0,0,0,1,4,0,0,1,23,31,365,1,12,4,0,0.21864


## test standardizing the data with the standard scaler

In [195]:
from sklearn.preprocessing import StandardScaler
def normlize_standard(x,y=None):
    scaler=StandardScaler()
    x=scaler.fit_transform(x)
    if y is not None:
        y=scaler.fit_transform(y)
        return x, y
    else:
        return x

## test standardizing the data with the Min Max Scaler

In [196]:
from sklearn.preprocessing import MinMaxScaler
def normalize_minMax(x, y=None):
    scaler=MinMaxScaler()
    x=scaler.fit_transform(x)
    if y is not None:
        y=scaler.fit_transform(y)
        return x,y 
    else:
        return x

### Prepare cross-validation model


In [197]:
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error, root_mean_squared_error
def model(mod, x_train, x_test, y_train, y_test, metric):
    mod.fit(x_train, y_train)
    predicted=mod.predict(x_test)
    return metric(y_test, predicted)
        # r2_score(y_test, predicted)
        # mean_absolute_error(y_test, predicted), 
        # mean_squared_error(y_test, predicted), 
        # root_mean_squared_error(y_test, predicted)

In [198]:
from sklearn.model_selection import train_test_split, StratifiedKFold, KFold, cross_val_score
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.utils import shuffle
import xgboost as xg
def cross_folding(x,y, metric):
    kf=KFold(n_splits=5, shuffle=True)
    arr=[]
    lf=pd.DataFrame()
    for i, (train,test) in enumerate(kf.split(x,y)):
        x_train, x_test, y_train, y_test=x[train], x[test], y[train], y[test]
        LR=model(LinearRegression(), x_train, x_test, y_train, y_test, metric)
        RF=model(RandomForestRegressor(n_estimators=3, max_depth=2, n_jobs=-1), x_train, x_test, y_train, y_test, metric)
        XG=model(xg.XGBRegressor(learning_rate= 0.1, max_depth= 7, subsample= 0.7), x_train, x_test, y_train, y_test, metric)
        frame=pd.DataFrame({
        "Split Number": [i],
        "Linear Regression": [LR],
        "Random Forest": [RF],
        "XG Boost": [XG]
        })
        arr.append(frame)
    lf=pd.concat(arr)
    return lf.mean()

In [199]:
def without_preprocessing(d):
    x,y=prepare_observations(d)
    average=cross_folding(x, y, r2_score)
    return average

In [200]:
def with_standardScalar(d):
    x,y=prepare_observations(d)
    x=normlize_standard(x)
    average=cross_folding(x, y, r2_score)
    return average

In [201]:
def with_minMaxScalar(d):
    x,y=prepare_observations(d)
    x=normalize_minMax(x)
    average=cross_folding(x, y, r2_score)
    return average

In [202]:
def apply_modelling(d):
    average_0=without_preprocessing(d)
    average_1=with_standardScalar(d)
    average_2=with_minMaxScalar(d)   
    return pd.concat([average_0, average_1, average_2], axis=1, keys=['no normalization','standard Scalar','MinMaxScalar'])

In [203]:
apply_modelling(df)

,no normalization,standard Scalar,MinMaxScalar
Split Number,2.000000,2.000000,2.000000
Linear Regression,0.430795,0.431226,0.432402
Random Forest,0.242231,0.243725,0.243935
XG Boost,0.820259,0.834247,0.835252


In [18]:
x, y=prepare_observations(df)
len(y.columns)

29

---------------------

### Tuning XGboost Model

In [28]:
from sklearn.model_selection import GridSearchCV
def tunning(training_data, testing_data, model, param):
        x_train, x_test, y_train, y_test=train_test_split(x,y, test_size=0.3)
        grid_search = GridSearchCV(model, param_grid=param, cv=KFold(3), 
                                   scoring='r2',
                                   n_jobs=-1)
        grid_search.fit(x_train, y_train)
        return grid_search.best_params_, grid_search.best_score_

In [49]:
from xgboost import XGBRegressor
from sklearn.model_selection import train_test_split
params= {
    'max_depth': [3, 5, 7],
    'learning_rate': [0.1, 0.01, 0.001],
    'subsample': [0.5, 0.7, 1]
}
x,y=prepare_observations(df)
best_param, best_score=tunning(x,y, XGBRegressor(), params)

NameError: name 'tunning' is not defined

In [405]:
x_train, x_test, y_train, y_test=train_test_split(x, y, test_size=0.3, random_state=1)
XG=model(XGBRegressor(learning_rate= 0.1, max_depth= 7, subsample= 0.7), x_train, x_test, y_train, y_test, r2_score)
XG

0.917816162109375

### Deep Learning with pytorch

In [19]:
import torch
import torch.nn as nn
import torch.nn.functional as tf

In [20]:
import gc
torch.cuda.empty_cache()
gc.collect()

4315

In [54]:
class Model(nn.Module):
    def __init__(self, features_num=23, h1=7, h2=8, out_features=29):
        super().__init__()
        self.fc1 = nn.Linear(features_num, h1)
        self.fc2 = nn.Linear(h1, h2)
        self.fc3 = nn.Linear(h2, out_features)
    def forward(self, x):
        x=tf.relu(self.fc1(x))
        x=tf.relu(self.fc2(x))
        x=self.fc3(x)
        return x

In [55]:
from sklearn.model_selection import train_test_split
model=Model()
x,y=prepare_observations(df)
x,y=normlize_standard(x, y)
x_train, x_test, y_train, y_test=train_test_split(x, y, random_state=42, test_size=0.3)
x_train=torch.from_numpy(x_train).float()
x_test=torch.from_numpy(x_test).float()
y_train=torch.from_numpy(y_train).float()
y_test=torch.from_numpy(y_test).float()

In [56]:
## Creating a criterion to measure the error of the model
criterion=nn.MSELoss()

In [57]:
optimizer=torch.optim.Adam(model.parameters(),lr=0.01)

In [58]:
# from torch.utils.data import TensorDataset, DataLoader
# data_set=TensorDataset(x_train, y_train)
# dataset=DataLoader(dataset=data_set, batch_size=2048, shuffle=True)

In [ ]:
epochs=10000
losses=[]
for i in range(epochs):
    # for x, y in dataset:
    y_pred=model.forward(x_train)
    loss=criterion(y_pred, y_train)
    losses.append(loss)
    loss.backward()
    optimizer.step()
    optimizer.zero_grad()
    if i %10 ==0:
        print(f"{i} iteration, loss: {loss}")

## Calculating measures

In [ ]:
# prepare measurement
Test_data=x.head(1)
Actual_data=y.row(1)

In [52]:
Test_data

NameError: name 'Test_data' is not defined

In [ ]:
Actual_data

In [ ]:
# Select the best model 
import xgboost as xg
from sklearn.model_selection import train_test_split
x,x_test, y, y_test= train_test_split(x,y, test_size=.3, random_state=42)
xg_model=xg.XGBRegressor()
xg_model.fit(x,y)
estimated_values=xg_model.predict(Test_data)

In [ ]:
estimated_values.flatten()

## Observation

Apparently using r square as a metric to represent to the data is a misrepresentation to the model's performance.
R square formula is - >1- ( total of squared residulas /total of squared standard deviation)
Knowing MAE is the total of residuals -> total(y - yPredicted)
giving r square = 0.02 therefore
giving standard deviation of .001 as given above
then r square= .0004 / .00001= 40 -> 1-40 =-39 which is wrong

Therefore, MAE score is the perfect representation of the data.
another testing metric should be provided.

Input data = timestamp Total      Usage     Climate Zone.
             1545212700000000     1.22827       1

Actual result=(0.1280200034379959, 0.0, 0.0, 0.01245999988168478, 0.0).


Estimated result= [2.9516332e-02, 4.0768548e-03, 1.0928856e-17, 1.0773698e-02,
       2.7552096e-03].